# Smart Retail & Customer Intelligence Platform — Module C (Colab runner)

This notebook:
1. Mounts your Google Drive (where your trained models live).
2. Clones/copies the `smart-retail-ai` project code into the Colab runtime.
3. Copies your trained model files from Drive into `app/models/`.
4. Installs dependencies.
5. Runs the FastAPI app **live** using `ngrok`, so you get a public URL + Swagger docs (`/docs`) you can hit from Postman or a browser, right from Colab.

**Before running:** update `DRIVE_PROJECT_FOLDER` below to point at wherever you keep
`product_classifier.h5`, `face_db.pkl`, `sentiment_model.pkl`, `vectorizer.pkl`,
`chatbot_model.pkl` in your Drive (e.g. the folder you shared).

In [ ]:
# 1. Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
PROJECT_DIR = "/content/drive/MyDrive/smart-retail-ai"
os.makedirs(f"{PROJECT_DIR}/app/services", exist_ok=True)
os.makedirs(f"{PROJECT_DIR}/app/models", exist_ok=True)
os.makedirs(f"{PROJECT_DIR}/data", exist_ok=True)
print("Project dir:", PROJECT_DIR)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Project dir: /content/drive/MyDrive/smart-retail-ai


In [ ]:
import os
import shutil

# Create local data folder
os.makedirs("data", exist_ok=True)

# Google Drive path
data = "/content/drive/MyDrive/smart-retail-ai/data"
drive_intents = os.path.join(data, "intents.json")

if os.path.exists(drive_intents):
    shutil.copy(drive_intents, "data/intents.json")
    print("Copied your custom intents.json")
else:
    print("Using the sample data/intents.json shipped with the project")

Copied your custom intents.json


In [ ]:
# 6. Install dependencies (first run only — takes a few minutes for dlib/tensorflow)
%cd /content/drive/MyDrive/smart-retail-ai
!pip install -q -r requirements.txt

/content/drive/MyDrive/smart-retail-ai


In [ ]:
file_path = "/content/drive/MyDrive/smart-retail-ai/app/services/chatbot_service.py"

with open(file_path, "r") as f:
    text = f.read()

text = text.replace(
    "from nlp_utils import preprocess",
    "from .nlp_utils import preprocess"
)

with open(file_path, "w") as f:
    f.write(text)

print("Fixed import")

Fixed import


In [ ]:
!grep -n "preprocess" /content/drive/MyDrive/smart-retail-ai/app/services/chatbot_service.py

25:from .nlp_utils import preprocess
48:                cleaned = preprocess(pattern)
120:        cleaned = preprocess(message)


In [ ]:
import sys
import importlib

sys.path.insert(
    0,
    "/content/drive/MyDrive/smart-retail-ai"
)

chatbot_service = importlib.import_module(
    "app.services.chatbot_service"
)

print("Chatbot service imported successfully")

Chatbot service imported successfully


## 8. Expose the API live from Colab

Two tunnel options — pick one:

- **ngrok** (needs a free authtoken from https://dashboard.ngrok.com — paste it below)
- **localtunnel** (no signup, sometimes less stable) — see the alternate cell after

Either way, this runs `uvicorn` **in a background thread** inside the notebook process, so the cell returns immediately and the server keeps running until you stop it or the runtime disconnects.

In [32]:
# 8a. Run + expose via ngrok
!pip install -q pyngrok nest_asyncio

import nest_asyncio
nest_asyncio.apply()

from app.services import cv_utils, nlp_utils, chatbot_service

from pyngrok import ngrok
import uvicorn
import threading

NGROK_AUTHTOKEN = ''  # <-- paste your ngrok authtoken here (from dashboard.ngrok.com)
if NGROK_AUTHTOKEN:
    ngrok.set_auth_token(NGROK_AUTHTOKEN)

# Optional: set an API key to simulate production auth (leave blank to disable auth for easy testing)
import os
os.environ['API_KEY'] = ''  # e.g. 'my-secret-key' — then send header X-API-Key: my-secret-key

public_url = ngrok.connect(8000)
print('Public URL:', public_url)
print('Swagger docs:', f'{public_url}/docs')

from app.main import app as fastapi_app

def run():
    uvicorn.run(fastapi_app, host='0.0.0.0', port=8000)

thread = threading.Thread(target=run, daemon=True)
thread.start()

Public URL: NgrokTunnel: "https://destitute-pope-distrust.ngrok-free.dev" -> "http://localhost:8000"
Swagger docs: NgrokTunnel: "https://destitute-pope-distrust.ngrok-free.dev" -> "http://localhost:8000"/docs


In [ ]:
# 8b. Alternative: localtunnel (no signup needed) — use INSTEAD of 8a, not in addition
# !npm install -g localtunnel -q
# import nest_asyncio, threading, uvicorn
# nest_asyncio.apply()
# from app.main import app as fastapi_app
# def run():
#     uvicorn.run(fastapi_app, host='0.0.0.0', port=8000)
# threading.Thread(target=run, daemon=True).start()
# !npx localtunnel --port 8000

In [42]:
public_url = ngrok.connect(8000)
print(public_url.public_url)

https://destitute-pope-distrust.ngrok-free.dev


In [44]:
import requests

BASE = public_url.public_url

# Helper function to handle API responses
def handle_response(response):
    if response.ok:
        return response.json()
    else:
        print(f"API request failed with status code: {response.status_code}")
        print(f"Response text: {response.text}")
        return None

# Chatbot
r = requests.post(
    f"{BASE}/chatbot",
    json={"message": "what is your return policy?"}
)
chatbot_response = handle_response(r)
if chatbot_response:
    print("Chatbot response:", chatbot_response)

# Sentiment
r = requests.post(
    f"{BASE}/analyze-sentiment",
    json={"text": "This product exceeded my expectations!"}
)
sentiment_response = handle_response(r)
if sentiment_response:
    print("Sentiment response:", sentiment_response)

# Dashboard
r = requests.get(f"{BASE}/dashboard/stats")
dashboard_response = handle_response(r)
if dashboard_response:
    print("Dashboard stats:", dashboard_response)

API request failed with status code: 502
Response text: <!DOCTYPE html>
<html class="h-full" lang="en-US" dir="ltr">
  <head>
    <meta charset="utf-8">
    <meta name="viewport" content="width=device-width, initial-scale=1">
    <link rel="preload" href="https://assets.ngrok.com/fonts/euclid-square/EuclidSquare-Regular-WebS.woff" as="font" type="font/woff" crossorigin="anonymous" />
    <link rel="preload" href="https://assets.ngrok.com/fonts/euclid-square/EuclidSquare-RegularItalic-WebS.woff" as="font" type="font/woff" crossorigin="anonymous" />
    <link rel="preload" href="https://assets.ngrok.com/fonts/euclid-square/EuclidSquare-Medium-WebS.woff" as="font" type="font/woff" crossorigin="anonymous" />
    <link rel="preload" href="https://assets.ngrok.com/fonts/euclid-square/EuclidSquare-MediumItalic-WebS.woff" as="font" type="font/woff" crossorigin="anonymous" />
    <link rel="preload" href="https://assets.ngrok.com/fonts/ibm-plex-mono/IBMPlexMono-Text.woff" as="font" type="font/w

API request failed with status code: 502
Response text: <!DOCTYPE html>
<html class="h-full" lang="en-US" dir="ltr">
  <head>
    <meta charset="utf-8">
    <meta name="viewport" content="width=device-width, initial-scale=1">
    <link rel="preload" href="https://assets.ngrok.com/fonts/euclid-square/EuclidSquare-Regular-WebS.woff" as="font" type="font/woff" crossorigin="anonymous" />
    <link rel="preload" href="https://assets.ngrok.com/fonts/euclid-square/EuclidSquare-RegularItalic-WebS.woff" as="font" type="font/woff" crossorigin="anonymous" />
    <link rel="preload" href="https://assets.ngrok.com/fonts/euclid-square/EuclidSquare-Medium-WebS.woff" as="font" type="font/woff" crossorigin="anonymous" />
    <link rel="preload" href="https://assets.ngrok.com/fonts/euclid-square/EuclidSquare-MediumItalic-WebS.woff" as="font" type="font/woff" crossorigin="anonymous" />
    <link rel="preload" href="https://assets.ngrok.com/fonts/ibm-plex-mono/IBMPlexMono-Text.woff" as="font" type="font/w